# 11 - Packaging & Project Structure

Concise, interview/revision focused. Part of Python & DSA. Builds a tiny real installable package end to end, in a scratch directory, so every claim below is verified rather than asserted.

# Part 1 - Why Packaging Matters

Without it: import paths break outside one specific folder, dependencies are undocumented, versions are not pinned (so "works on my machine" is common), and there is no clean way to `pip install` your own code elsewhere. pyproject.toml is the modern, standardized answer, replacing the older setup.py/setup.cfg split.

### Standard project layout: src layout vs flat layout

```
flat layout                  src layout (preferred for anything you will publish/install)
my_project/                  my_project/
  my_package/                  src/
    __init__.py                  my_package/
    core.py                        __init__.py
  tests/                            core.py
  pyproject.toml                tests/
                                 pyproject.toml
```

src layout forces tests to run against the INSTALLED package, not whatever happens to be in the current directory -- catches "forgot to add this file to the package" bugs that a flat layout can silently hide during local development.

# Part 2 - Building a Real, Installable Package

A minimal pyproject.toml, an actual package, and a real `pip install -e .` (editable install) -- verified below, not just described.

In [1]:
import pathlib, textwrap, subprocess, sys, shutil

root = pathlib.Path("/tmp/demo_pkg")
shutil.rmtree(root, ignore_errors=True)
(root / "src" / "demo_pkg").mkdir(parents=True)
(root / "tests").mkdir()

(root / "pyproject.toml").write_text(textwrap.dedent('''
    [build-system]
    requires = ["setuptools>=68"]
    build-backend = "setuptools.build_meta"

    [project]
    name = "demo-pkg"
    version = "0.1.0"
    description = "a tiny demo package"
    requires-python = ">=3.9"
    dependencies = [
        "requests>=2.28",
    ]

    [project.optional-dependencies]
    dev = ["pytest>=7.0"]

    [project.scripts]
    demo-cli = "demo_pkg.cli:main"

    [tool.setuptools.packages.find]
    where = ["src"]
'''))

(root / "src" / "demo_pkg" / "__init__.py").write_text('from .core import add\n__all__ = ["add"]\n')
(root / "src" / "demo_pkg" / "core.py").write_text("def add(a, b):\n    return a + b\n")
(root / "src" / "demo_pkg" / "cli.py").write_text(
    "from .core import add\n\ndef main():\n    print('2 + 3 =', add(2, 3))\n"
)
print(sorted(str(p.relative_to(root)) for p in root.rglob("*") if p.is_file()))

['pyproject.toml', 'src/demo_pkg/__init__.py', 'src/demo_pkg/cli.py', 'src/demo_pkg/core.py']


### pyproject.toml sections explained

- `[build-system]` -- which tool actually builds the package (setuptools, hatchling, poetry-core, ...) and how pip should invoke it.
- `[project]` -- name, version, and `dependencies` -- the REQUIRED packages installed automatically whenever this package is installed.
- `[project.optional-dependencies]` -- extras, e.g. `pip install demo-pkg[dev]` pulls in pytest too; normal installs skip them.
- `[project.scripts]` -- registers a real shell command (`demo-cli`) that runs `main()` -- this is how `black`, `pytest`, `uvicorn` etc. become terminal commands after installing.

In [2]:
result = subprocess.run(
    # --break-system-packages is only needed because this notebook's own interpreter is a
    # system Python, not an activated venv (PEP 668 blocks pip otherwise). Inside a normal
    # venv, as recommended in Part 3, plain `pip install -e .` works with no extra flag.
    [sys.executable, "-m", "pip", "install", "-e", str(root), "-q", "--break-system-packages"],
    capture_output=True, text=True,
)
print("install exit code:", result.returncode)
print(result.stderr[-500:] if result.returncode else "installed cleanly")

install exit code: 0
installed cleanly


### A real gotcha: importing a package just installed in THIS SAME process

A freshly-installed editable package is not automatically importable in a process that was already running before the install -- the interpreter only processes site-packages' `.pth`/import-hook files once, at startup. Normally this is invisible, because you install packages BEFORE starting the process that uses them (or in a fresh terminal). It only bites in exactly this situation: installing from inside a long-lived process, such as this notebook kernel. The fix is `site.addsitedir()` to re-process site-packages, plus `importlib.invalidate_caches()`.

In [3]:
import site, importlib
for d in site.getsitepackages():
    site.addsitedir(d)
importlib.invalidate_caches()

import demo_pkg
print("demo_pkg.add(2, 3) =", demo_pkg.add(2, 3))

# the console script from [project.scripts] is now a real, callable command
r = subprocess.run(["demo-cli"], capture_output=True, text=True)
print("demo-cli output:", r.stdout.strip())

demo_pkg.add(2, 3) = 5
demo-cli output: 2 + 3 = 5


### Editable install: why -e matters during development

`pip install .` copies the package into site-packages -- editing source afterward has no effect until reinstalled. `pip install -e .` instead links back to the source directory, so edits are visible immediately without reinstalling, which is why every real project is installed editable during development.

In [4]:
(root / "src" / "demo_pkg" / "core.py").write_text("def add(a, b):\n    return a + b + 100  # changed\n")

importlib.reload(demo_pkg.core)
importlib.reload(demo_pkg)
print("after editing source, no reinstall:", demo_pkg.add(2, 3))   # reflects the edit immediately

after editing source, no reinstall: 105


# Part 3 - Virtual Environments & Dependency Files

### venv: an isolated interpreter + package set per project

Without it, every project on a machine shares one global site-packages -- two projects needing different versions of the same library cannot coexist. The commands (not run here, since this notebook's own kernel is itself a specific interpreter):

```bash
python3 -m venv .venv                 # create
source .venv/bin/activate             # activate (Linux/macOS)
.venv\Scripts\activate                # activate (Windows)
pip install -r requirements.txt       # or: pip install -e ".[dev]"
deactivate                            # leave it
```

### requirements.txt vs pyproject.toml dependencies vs a lockfile

- `pyproject.toml [project.dependencies]` -- what your package NEEDS, usually with loose version ranges (`>=2.28`), so it stays installable alongside other packages.
- `requirements.txt` -- traditionally a flat, often exact-pinned list (`requests==2.31.0`) for reproducing ONE specific environment, such as a deployment.
- A lockfile (`poetry.lock`, `uv.lock`, `pip-compile`'s output) -- exact, hash-verified versions of every dependency AND every transitive sub-dependency, for fully reproducible installs. Loose ranges in pyproject.toml plus a lockfile is the modern combination: flexible for the package, exact for any given install.

In [5]:
pip_freeze = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True)
print(pip_freeze.stdout.count("\n"), "packages pinned by `pip freeze` in the CURRENT environment")
print("\n".join(pip_freeze.stdout.splitlines()[:5]), "\n...")
print("\nthis is exactly what a hand-maintained requirements.txt captures: exact versions, no ranges")

187 packages pinned by `pip freeze` in the CURRENT environment
absl-py==2.4.0
annotated-doc==0.0.5
annotated-types==0.8.0
anyio==4.14.2
argcomplete==3.1.4 
...

this is exactly what a hand-maintained requirements.txt captures: exact versions, no ranges


## Interview rapid-fire

- `__init__.py` marks a directory as a regular package (pre-3.3) and still controls what `from package import *` exposes and what runs on first import -- namespace packages (no `__init__.py`) exist but are less common for applications.
- Absolute imports (`from demo_pkg.core import add`) are preferred over relative (`from .core import add`) outside the package itself; relative imports are fine and normal WITHIN a package's own modules.
- `pip install .` vs `pip install -e .`: the difference is a copy vs a link back to source -- only `-e` reflects live source edits without reinstalling.
- A pinned `requirements.txt` guarantees reproducibility for one environment; a `pyproject.toml` with ranges keeps the package installable alongside others -- libraries ship the latter, deployed applications typically add the former (or a lockfile) on top.

## Practice

1. Add a `[project.optional-dependencies]` group called `test` containing `pytest` and `httpx`, and install it with `pip install -e ".[test]"`.
2. Add a second module `utils.py` to `demo_pkg`, import it from `__init__.py`, and confirm it is importable as `demo_pkg.utils`.
3. Write a `requirements.txt` by hand for a project depending on `fastapi` and `uvicorn`, pinned to exact versions, and explain when you would choose that over letting `pyproject.toml` ranges resolve freely.